# DriveNet — entrenamiento en GPU (Google Colab / Kaggle)

Entrena la red de dirección end-to-end del vehículo TMR 2026 (clonación de
comportamiento, `TMR2026/vision/drive_net.py`) en una GPU gratuita y la exporta a
formatos ligeros para la Raspberry Pi 5.

**Qué hace este notebook**

1. Verifica la GPU y clona el repositorio.
2. Monta el dataset (Google Drive, Kaggle Input o un `.zip`) — o genera uno sintético de prueba.
3. Inspecciona etiquetas y muestra el aumento de datos (brillo, contraste, sombras, reflejos, desplazamiento lateral).
4. Entrena **PilotNet** o **MobileNet ligera** con *early stopping*, precisión mixta (AMP) y balanceo de curvas.
5. Grafica pérdida / MSE / RMSE / MAE de validación.
6. Evalúa sobre el set de validación.
7. Exporta a ONNX FP32/FP16/INT8 y TFLite FP16/INT8, verifica paridad contra PyTorch.
8. Guarda pesos y exportaciones en Drive / descarga un `.zip`.

**Formato del dataset (tub)** — el que escriben `tools/record_driving.py` y `tools/gen_synth_driving.py`:

```
<tub>/frames/000000.jpg, 000001.jpg, ...
<tub>/labels.csv      frame,error_px,confidence,source[,throttle]
```

`error_px` = centro del carril − centro de la imagen (px, positivo = el carril está a la derecha).
Es la misma señal que produce `LanePipeline`, así que la red sustituye al seguidor clásico
sin tocar FSM, PID ni luces.

> En Colab: *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*.
> En Kaggle: *Settings → Accelerator → GPU*, e *Internet → On* para clonar.

## 1. GPU y entorno

In [ ]:
import os, sys, subprocess
IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if IN_KAGGLE else ("/content" if IN_COLAB else os.getcwd())
print("Colab:", IN_COLAB, "| Kaggle:", IN_KAGGLE, "| WORK:", WORK)

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("AVISO: sin GPU — el entrenamiento funcionará, pero lento.")

## 2. Clonar el repositorio
Si el repo es privado, crea un *token* de GitHub de solo lectura y pásalo en `GITHUB_TOKEN` (en Colab: *Secrets*; en Kaggle: *Add-ons → Secrets*).

In [ ]:
REPO   = "Empanaditasesina72/Sim2Real-Scale-Vehicle"
BRANCH = "main"        # o la rama del PR si aún no se ha fusionado
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")
try:                                   # Colab Secrets (opcional)
    from google.colab import userdata
    GITHUB_TOKEN = GITHUB_TOKEN or (userdata.get("GITHUB_TOKEN") or "")
except Exception:
    pass

REPO_DIR = os.path.join(WORK, "Sim2Real-Scale-Vehicle")
if not os.path.exists(REPO_DIR):
    auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH,
                    f"https://{auth}github.com/{REPO}.git", REPO_DIR], check=True)
TMR = os.path.join(REPO_DIR, "TMR2026")
os.chdir(TMR)
sys.path.insert(0, TMR)
print("Repo en", TMR)

## 3. Dependencias
Solo lo que falta en la imagen de Colab/Kaggle. La conversión a TFLite (`onnx2tf`) es opcional y pesada.

In [ ]:
%pip -q install onnx onnxruntime onnxscript onnxconverter-common opencv-python-headless matplotlib
WANT_TFLITE = True   # pon False si no necesitas .tflite
if WANT_TFLITE:
    %pip -q install onnx2tf onnx_graphsurgeon sng4onnx tf_keras

## 4. Dataset

Elige **una** fuente en `DATA_SOURCE`:

| valor | qué hace |
|---|---|
| `"drive"` | monta Google Drive y descomprime `DRIVE_ZIP` (un `.zip` con uno o más tubs) |
| `"kaggle"` | usa los tubs de `/kaggle/input/<tu-dataset>` |
| `"zip"` | descomprime un `.zip` ya subido a `WORK` |
| `"synthetic"` | genera carreteras sintéticas (sirve para probar todo el pipeline sin datos reales) |

Cualquier carpeta que contenga `labels.csv` se toma como un tub.

In [ ]:
DATA_SOURCE = "synthetic"          # "drive" | "kaggle" | "zip" | "synthetic"
DRIVE_ZIP   = "/content/drive/MyDrive/TMR2026/drive_tubs.zip"
KAGGLE_DIR  = "/kaggle/input"
LOCAL_ZIP   = os.path.join(WORK, "drive_tubs.zip")
DATA_DIR    = os.path.join(WORK, "datasets")
os.makedirs(DATA_DIR, exist_ok=True)

import zipfile, glob
if DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    zipfile.ZipFile(DRIVE_ZIP).extractall(DATA_DIR)
elif DATA_SOURCE == "zip":
    zipfile.ZipFile(LOCAL_ZIP).extractall(DATA_DIR)
elif DATA_SOURCE == "kaggle":
    DATA_DIR = KAGGLE_DIR
elif DATA_SOURCE == "synthetic":
    for name, n, seed in (("synth_train", 6000, 0), ("synth_val", 1000, 999)):
        out = os.path.join(DATA_DIR, name)
        if not os.path.exists(os.path.join(out, "labels.csv")):
            subprocess.run([sys.executable, "tools/gen_synth_driving.py",
                            "--n", str(n), "--out", out, "--seed", str(seed)], check=True)

TUBS = sorted({os.path.dirname(p) for p in glob.glob(os.path.join(DATA_DIR, "**", "labels.csv"), recursive=True)})
print(f"{len(TUBS)} tubs encontrados:")
for t in TUBS: print("  ", t)
assert TUBS, "No hay tubs (carpetas con labels.csv) en " + DATA_DIR

### Separación entrenamiento / validación
Lo ideal es validar con **tubs completos distintos** (otra vuelta, otra luz), no con frames
vecinos del mismo recorrido. Pon en `VAL_TUBS` los tubs de validación; si lo dejas vacío,
se separa en bloques contiguos de 50 frames (`--val-split`), que evita la fuga entre frames consecutivos.

In [ ]:
VAL_TUBS   = [t for t in TUBS if "val" in os.path.basename(t).lower()]
TRAIN_TUBS = [t for t in TUBS if t not in VAL_TUBS]
print("train:", [os.path.basename(t) for t in TRAIN_TUBS])
print("val  :", [os.path.basename(t) for t in VAL_TUBS] or "(split automático)")

## 5. Inspección de etiquetas

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from vision.drive_data import read_tubs, balance_weights
NORM_PX = 150.0
s = read_tubs(TRAIN_TUBS)
err = np.array([x.error_px for x in s])
print(f"{len(s)} frames | error_px media {err.mean():+.1f}  std {err.std():.1f}  "
      f"min {err.min():.0f}  max {err.max():.0f}")
w = balance_weights(s, NORM_PX)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].hist(err, bins=60); ax[0].set_title("error_px (crudo)"); ax[0].set_xlabel("px")
ax[1].hist(err, bins=60, weights=w * len(w)); ax[1].set_title("error_px con --balance")
plt.show()

## 6. Vista previa del aumento de datos
La fila de arriba es el frame original; las demás, variantes aumentadas con su etiqueta corregida (la línea verde marca el centro del carril). Abajo, el recorte ROI que recibe la red.

In [ ]:
import cv2
from vision.drive_data import augment, AugConfig
from vision.drive_net import preprocess
rng = np.random.default_rng(0)
idx = rng.choice(len(s), 4, replace=False)
fig, axes = plt.subplots(5, 4, figsize=(16, 13))
for c, i in enumerate(idx):
    img0 = cv2.imread(s[i].path)
    for r in range(4):
        img, e = (img0, s[i].error_px) if r == 0 else augment(img0, s[i].error_px, rng, AugConfig())
        vis = img.copy(); h, wd = vis.shape[:2]
        x = int(np.clip(wd / 2 + e, 0, wd - 1))
        cv2.line(vis, (x, h), (x, h // 2), (0, 255, 0), 3)
        axes[r, c].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); axes[r, c].axis("off")
        axes[r, c].set_title(("original " if r == 0 else "") + f"err {e:+.0f}px", fontsize=9)
    roi = preprocess(img0)                       # (3,H,W) YUV/255
    axes[4, c].imshow(cv2.cvtColor((roi.transpose(1, 2, 0) * 255).astype(np.uint8), cv2.COLOR_YUV2RGB))
    axes[4, c].set_title("ROI -> red (66x200)", fontsize=9); axes[4, c].axis("off")
plt.tight_layout(); plt.show()

## 7. Hiperparámetros

| parámetro | qué controla |
|---|---|
| `ARCH` | `"pilotnet"` (NVIDIA, ~0.25 M parámetros) o `"mobilenet"` (MobileNetV2 ligera, ~0.17 M, cuantiza bien a INT8) |
| `SMOOTH` | ventana de media móvil sobre etiquetas **grabadas** (no se aplica a las sintéticas). 5 ≈ 0.17 s a 30 FPS |
| `BALANCE` | sobremuestrea curvas (|error| grande) para que la red no aprenda solo "ir recto" |
| `SHIFT_GAIN` | px de etiqueta por px de imagen en el aumento por desplazamiento lateral: 1.0 para sintéticos; para tubs etiquetados por `LanePipeline`, la escala BEV/imagen |
| `THROTTLE` | entrena además una cabeza de velocidad si los tubs traen la columna `throttle` |
| `PATIENCE` | *early stopping*: épocas sin mejorar `MIN_DELTA` px de RMSE antes de parar |

In [ ]:
ARCH       = "pilotnet"     # "pilotnet" | "mobilenet"
EPOCHS     = 60
BATCH      = 256 if torch.cuda.is_available() else 64
LR         = 1e-3
LOSS       = "huber"        # "huber" | "mse"
SMOOTH     = 0              # 5 para datos grabados en la pista
BALANCE    = True
SHIFT_GAIN = 1.0
THROTTLE   = False
PATIENCE   = 10
MIN_DELTA  = 0.5
WORKERS    = 2
OUT_DIR    = os.path.join(WORK, "drive_out")
os.makedirs(OUT_DIR, exist_ok=True)
WEIGHTS    = os.path.join(OUT_DIR, "drive_net.pt")

## 8. Entrenamiento
Los mejores pesos se guardan en disco **cada vez que mejora** la validación, así que si Colab corta la sesión no se pierden.

In [ ]:
from tools.train_drive import build_parser, train
argv = ["--data", ",".join(TRAIN_TUBS), "--arch", ARCH, "--epochs", str(EPOCHS),
        "--batch", str(BATCH), "--lr", str(LR), "--loss", LOSS,
        "--norm-px", str(NORM_PX), "--smooth", str(SMOOTH),
        "--shift-gain", str(SHIFT_GAIN), "--patience", str(PATIENCE),
        "--min-delta", str(MIN_DELTA), "--workers", str(WORKERS), "--out", WEIGHTS]
if VAL_TUBS: argv += ["--val", ",".join(VAL_TUBS)]
if BALANCE:  argv += ["--balance"]
if THROTTLE: argv += ["--throttle"]
hist = train(build_parser().parse_args(argv))

## 9. Curvas de pérdida y métricas

In [ ]:
ep = np.arange(1, len(hist["train_loss"]) + 1)
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
ax[0].plot(ep, hist["train_loss"]); ax[0].set_yscale("log")
ax[0].set_title(f"pérdida de entrenamiento ({LOSS})"); ax[0].set_xlabel("época")
ax[1].plot(ep, hist["val_mse_px2"], "r"); ax[1].set_yscale("log")
ax[1].set_title("MSE validación (px²)"); ax[1].set_xlabel("época")
ax[2].plot(ep, hist["val_rmse_px"], "r", label="RMSE")
ax[2].plot(ep, hist["val_mae_px"], "r--", label="MAE")
ax[2].axvline(hist["best_epoch"], color="k", ls=":", label=f"mejor ({hist['best_epoch']})")
ax[2].set_title("error de validación (px)"); ax[2].legend(); ax[2].set_xlabel("época")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "curvas.png"), dpi=150); plt.show()
print(f"Mejor RMSE val: {min(hist['val_rmse_px']):.1f} px en la época {hist['best_epoch']}")

## 10. Evaluación sobre validación
RMSE contra las etiquetas y un mosaico: línea amarilla = etiqueta, verde = predicción.

In [ ]:
val_for_eval = VAL_TUBS[0] if VAL_TUBS else TRAIN_TUBS[0]
mont = os.path.join(OUT_DIR, "drive_eval.png")
subprocess.run([sys.executable, "tools/test_drive_net.py", "--weights", WEIGHTS,
                "--tub", val_for_eval, "--out", mont], check=True)
plt.figure(figsize=(16, 9)); plt.imshow(plt.imread(mont)); plt.axis("off"); plt.show()

## 11. Exportación a formatos ligeros

* **ONNX FP32** — referencia.
* **ONNX FP16** — mitad de tamaño.
* **ONNX INT8** — cuantización estática por canal, **calibrada con frames reales** del dataset: la vía más rápida en la CPU de la Pi 5 con `onnxruntime`.
* **TFLite FP16 / INT8** — vía `onnx2tf`, para `ai-edge-litert` / `tflite-runtime`.

La tabla final da el error de cada exportación frente a PyTorch (px). Los tiempos medidos
aquí son de la CPU de Colab: **los válidos para el auto se miden en la Pi** con
`python tools/export_drive.py --bench` o `python tools/test_drive_net.py`.

In [ ]:
CALIB = ",".join(VAL_TUBS or TRAIN_TUBS)
cmd = [sys.executable, "tools/export_drive.py", "--weights", WEIGHTS,
       "--fp16", "--int8", "--calib", CALIB, "--bench", "--iters", "200"]
if WANT_TFLITE: cmd.append("--tflite")
subprocess.run(cmd, check=True)
!ls -la {OUT_DIR}

## 12. Guardar resultados
Copia todo a Google Drive (Colab) o déjalo en `/kaggle/working` (Kaggle lo conserva como *output*), y genera un `.zip` descargable.

In [ ]:
import shutil
zip_path = shutil.make_archive(os.path.join(WORK, "drive_net_export"), "zip", OUT_DIR)
print("ZIP:", zip_path)
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        dst = "/content/drive/MyDrive/TMR2026/drive_out"
        shutil.copytree(OUT_DIR, dst, dirs_exist_ok=True)
        print("Copiado a", dst)
        from google.colab import files
        files.download(zip_path)
    except Exception as e:
        print("Drive/descarga no disponible:", e)

## 13. Despliegue en la Raspberry Pi 5

```bash
# en el PC: copiar la exportación elegida al auto (Tailscale)
scp drive_net_int8.onnx drive_net.json angel01@100.68.26.120:~/Carrito/TMR2026/weights/

# en la Pi
pip install --break-system-packages onnxruntime
cd ~/Carrito/TMR2026
python tools/test_drive_net.py --weights weights/drive_net_int8.onnx --camera   # vista previa, sin motores
python tools/export_drive.py --weights weights/drive_net.pt --int8 --calib <tub> --bench  # FPS reales
```

Y en `TMR2026/config.py`:

```python
USE_DRIVE_NET     = True
DRIVE_NET_WEIGHTS = "weights/drive_net_int8.onnx"
```

`drive_net.json` debe acompañar al `.onnx`/`.tflite`: guarda el tamaño de entrada, la ROI y
`norm_px`, sin los cuales la predicción sale en una escala equivocada.
Si `DRIVE_NET_WEIGHTS` falta o falla al cargar, `main.py` vuelve solo al `LanePipeline` clásico.